# Aim
Modify the main pipeline:
- from: rather than storing large batches and pulling one message out)
- to: Instead, continuously store hexes, and store ~5min of them, to decode them together

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import planesailing as ps

from planesailing import main

import time
import datetime
import pandas as pd
import numpy as np
import scipy
from copy import copy

import cosmosdr
import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc

import structlog
logger = structlog.get_logger()

try:
    sdr.close()
except:
    pass

# import plotly
# from plotly.graph_objects import Scatter
# from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
# init_notebook_mode(connected=False)
# import plotly.express as px
# import plotly.graph_objects as go

pd.options.display.max_columns = 99

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2.4e6


# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us_after_upsampling = int(round(target_sr/1e6))  # 12

In [ ]:
n_reads_per_acquisition=64  # optimal for maximising parsed latlons
n_samples_per_read=4096  # optimal for maximising parsed latlons
sleep_time=0.01  # optimal for maximising parsed latlons
sample_rate=2.4e6
target_sr=12e6

In [ ]:
# class (message store):
# - ttl cache
# - f() compare against full cache function
# - f() insert msgs to cache


# pipeline:
# - 

# Q - when to decode? On final read (update UI?)

# Ensure any existing stream is stopped
s_acq.streamer.stop_stream()

# Kick off a stream
s_acq.streamer.start_stream(
    center_freq=1090e6,
    sample_rate=sample_rate,
    n_reads_per_acquisition=n_reads_per_acquisition,
    n_samples_per_read=n_samples_per_read,
    sleep_length_s=sleep_time
    / 2,  # Ensure the signal stream is updated more frequently than the reads happen
    sdr_gain="auto",
)

In [ ]:
hits = main._batch_generate_hexes(10, 0.01, verbose=False)

In [ ]:
hits

In [ ]:
main.RecentMessagesCache.cache.currsize

In [ ]:
df = main.decode_current_cached_messages()

df[~df.latitude.isna()]

# Now the architecture has shifted, grid search to see which params get the most hits in a single period of time

In [ ]:
n_reads_per_acquisition=64
n_samples_per_read=4096
sleep_time=0.01
n_reads_to_try_to_parse # radio

In [ ]:
3*3*3*3

In [ ]:
t0 = datetime.datetime.now()

In [ ]:
(datetime.datetime.now() - t0).seconds

In [ ]:
main._batch_generate_hexes(10, sleep_time, verbose=False)

In [ ]:
time_per_config = 5
results = []
for n_reads_per_acquisition in [4, 16, 64]:
    for n_samples_per_read in [2048, 4096, 4096*2]:
        for sleep_time in [0.001, 0.01, 0.1]:
            for n_reads_parse_ratio in [1, 2, 4]:

                n_reads_to_try_to_parse = int(n_reads_per_acquisition / n_reads_parse_ratio)
                
                # Ensure any existing stream is stopped
                s_acq.streamer.stop_stream()
                
                # Kick off a stream
                s_acq.streamer.start_stream(
                    center_freq=1090e6,
                    sample_rate=sample_rate,
                    n_reads_per_acquisition=n_reads_per_acquisition,
                    n_samples_per_read=n_samples_per_read,
                    sleep_length_s=sleep_time
                    / 2,  # Ensure the signal stream is updated more frequently than the reads happen
                    sdr_gain="auto",
                )
                # Give the streamer a moment to start up
                time.sleep(3)

                # clear cache
                main.RecentMessagesCache.cache.clear()
                
                # start timer
                t0 = datetime.datetime.now()
                time_passed = 0
                
                while not (time_passed > time_per_config):
                    main._batch_generate_hexes(10, sleep_time, verbose=False)

                    time_passed = (datetime.datetime.now() - t0).seconds
                                    
                res_n = main.RecentMessagesCache.cache.currsize
                # display(main.RecentMessagesCache.cache)
                
                # calculate msgs / sec
                res_per_sec = res_n / time_passed

                res = {
                    "n_reads_per_acquisition":n_reads_per_acquisition,
                    "n_samples_per_read":n_samples_per_read,
                    "sleep_time":sleep_time,
                    "n_reads_parse_ratio":n_reads_parse_ratio,
                    "n_reads_to_try_to_parse": n_reads_to_try_to_parse,
                    "res_n":res_n,
                    "res_per_sec":res_per_sec,
                }
                display(res)
                results.append(res)